In [ ]:
!pip -q install pandas requests matplotlib gradio

In [ ]:
import requests
import sqlite3
import pandas as pd
import re
import string
import time
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import unittest
import gradio as gr

In [ ]:
BASE_URL = "https://api.jikan.moe/v4"
DB_NAME = "anime_pipeline.db"

# You can change these
SEARCH_QUERY = "naruto"     # Example keyword search
MAX_PAGES = 3               # Keep moderate for demo
REQUEST_DELAY = 1.2

In [ ]:
def clean_text(text):
    """
    Clean text by:
    - converting to lowercase
    - removing punctuation
    - removing HTML tags
    - removing extra spaces
    """
    if text is None or pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_int(value, default=0):
    try:
        if value is None:
            return default
        return int(value)
    except:
        return default


def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except:
        return default


def classify_score_band(score):
    """
    Classifying score into bands.
    """

    if score < 6:
        return "low"
    elif score > 6 and score < 8:
        return "medium"
    else:
        return "high"


def classify_popularity_band(popularity):
    """
    Lower popularity number means more popular in MAL/Jikan ranking terms.
    """
    if popularity is None or popularity == 0:
        return "unknown"
    popularity = safe_int(popularity, 0)
    if popularity <= 1000:
        return "very popular"
    elif popularity <= 2000:
        return "popular"
    elif popularity <= 5000:
        return "moderate"
    else:
        return "not popular"


def classify_episode_group(episodes):
    if episodes is None or episodes == 0:
        return "unknown"
    episodes = safe_int(episodes, 0)
    if episodes <= 15:
        return "short"
    elif episodes <= 26:
        return "medium"
    else:
        return "long"


def extract_names_from_list(items):
    if not items:
        return []

    return [item["name"] for item in items if isinstance(item, dict) and "name" in item]


def count_items_in_csv(text):

    if not text:
        return 0
    return len([x.strip() for x in text.split(",") if x.strip()])


In [ ]:
def fetch_anime_search(query, max_pages=1, delay=2.0):
    """
    Fetch anime data from Jikan search endpoint.
    Endpoint example: /anime?q=naruto&page=1
    """
    all_anime = []

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/anime"
        params = {
            "q": query,
            "page": page
        }

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        page_data = data.get("data", [])
        if not page_data:
            break

        all_anime.extend(page_data)

        pagination = data.get("pagination", {})
        has_next_page = pagination.get("has_next_page", False)

        if not has_next_page:
            break

        time.sleep(delay)

    return all_anime


def fetch_top_anime(max_pages=1, delay=1.2):
    """
    Fetch top anime from Jikan top endpoint.
    """
    all_anime = []

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/top/anime"
        params = {
            "page": page
        }

        response = requests.get(url, params=params, timeout=60)
        data = response.json()

        page_data = data.get("data", [])
        if not page_data:
            break

        all_anime.extend(page_data)

        pagination = data.get("pagination", {})
        has_next_page = pagination.get("has_next_page", False)

        if not has_next_page:
            break

        time.sleep(delay)

    return all_anime


In [ ]:

#Logic for this function was self-devised and the code is AI-written for enhanced efficiency.
def normalize_anime_data(raw_anime, source_label="search"):
    """
    Convert raw Jikan response into a structured DataFrame.
    """
    rows = []

    for anime in raw_anime:
        genres = extract_names_from_list(anime.get("genres", []))
        themes = extract_names_from_list(anime.get("themes", []))
        studios = extract_names_from_list(anime.get("studios", []))
        demographics = extract_names_from_list(anime.get("demographics", []))

        aired = anime.get("aired", {})
        aired_from = aired.get("from")
        aired_to = aired.get("to")

        row = {
            "mal_id": anime.get("mal_id"),
            "source_label": source_label,
            "title": anime.get("title"),
            "title_english": anime.get("title_english"),
            "type": anime.get("type"),
            "source": anime.get("source"),
            "episodes": anime.get("episodes"),
            "status": anime.get("status"),
            "airing": anime.get("airing"),
            "score": anime.get("score"),
            "scored_by": anime.get("scored_by"),
            "rank": anime.get("rank"),
            "popularity": anime.get("popularity"),
            "members": anime.get("members"),
            "favorites": anime.get("favorites"),
            "synopsis": anime.get("synopsis"),
            "rating": anime.get("rating"),
            "year": anime.get("year"),
            "season": anime.get("season"),
            "aired_from": aired_from,
            "aired_to": aired_to,
            "genres": ", ".join(genres),
            "themes": ", ".join(themes),
            "studios": ", ".join(studios),
            "demographics": ", ".join(demographics),
            "url": anime.get("url"),
            "fetched_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        }
        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
#Logic for this function was self-devised and the code is AI-written for enhanced efficiency.
def preprocess_anime_data(df):
    """
    Clean and transform anime data.
    """
    if df.empty:
        return df.copy()

    processed = df.copy()

    # Remove duplicate anime using MAL id
    processed.drop_duplicates(subset=["mal_id"], inplace=True)

    # Clean text fields
    processed["clean_title"] = processed["title"].apply(clean_text)
    processed["clean_title_english"] = processed["title_english"].apply(clean_text)
    processed["clean_synopsis"] = processed["synopsis"].apply(clean_text)

    # Text lengths
    processed["title_length"] = processed["clean_title"].apply(len)
    processed["synopsis_length"] = processed["clean_synopsis"].apply(len)

    # Binary feature
    processed["has_english_title"] = processed["title_english"].apply(
        lambda x: 0 if x is None or str(x).strip() == "" else 1
    )

    # Safe numeric conversion
    numeric_cols = ["episodes", "score", "scored_by", "rank", "popularity", "members", "favorites", "year"]
    for col in numeric_cols:
        if col == "score":
            processed[col] = processed[col].apply(lambda x: safe_float(x, 0.0))
        else:
            processed[col] = processed[col].apply(lambda x: safe_int(x, 0))

    # Engineered features
    processed["score_band"] = processed["score"].apply(classify_score_band)
    processed["popularity_band"] = processed["popularity"].apply(classify_popularity_band)
    processed["episode_group"] = processed["episodes"].apply(classify_episode_group)

    processed["genre_count"] = processed["genres"].apply(count_items_in_csv)
    processed["theme_count"] = processed["themes"].apply(count_items_in_csv)
    processed["studio_count"] = processed["studios"].apply(count_items_in_csv)
    processed["demographic_count"] = processed["demographics"].apply(count_items_in_csv)

    return processed.reset_index(drop=True)

In [1]:
def create_database(db_name):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS raw_anime (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            mal_id INTEGER UNIQUE,
            source_label TEXT,
            title TEXT,
            title_english TEXT,
            type TEXT,
            source TEXT,
            episodes INTEGER,
            status TEXT,
            airing INTEGER,
            score REAL,
            scored_by INTEGER,
            rank_value INTEGER,
            popularity INTEGER,
            members INTEGER,
            favorites INTEGER,
            synopsis TEXT,
            rating TEXT,
            year INTEGER,
            season TEXT,
            aired_from TEXT,
            aired_to TEXT,
            genres TEXT,
            themes TEXT,
            studios TEXT,
            demographics TEXT,
            url TEXT,
            fetched_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS processed_anime (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            mal_id INTEGER,
            clean_title TEXT,
            clean_title_english TEXT,
            clean_synopsis TEXT,
            title_length INTEGER,
            synopsis_length INTEGER,
            has_english_title INTEGER,
            score_band TEXT,
            popularity_band TEXT,
            episode_group TEXT,
            genre_count INTEGER,
            theme_count INTEGER,
            studio_count INTEGER,
            demographic_count INTEGER,
            FOREIGN KEY(mal_id) REFERENCES raw_anime(mal_id)
        )
    """)

    conn.commit()
    conn.close()

In [2]:
def insert_raw_anime(df_raw, db_name):
    if df_raw.empty:
        return 0

    conn = sqlite3.connect(db_name)
    cur = conn.cursor()
    inserted = 0

    for _, row in df_raw.iterrows():
        try:
            cur.execute("""
                INSERT OR IGNORE INTO raw_anime (
                    mal_id, source_label, title, title_english, type, source, episodes, status,
                    airing, score, scored_by, rank_value, popularity, members, favorites,
                    synopsis, rating, year, season, aired_from, aired_to, genres, themes,
                    studios, demographics, url, fetched_at
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row["mal_id"], row["source_label"], row["title"], row["title_english"], row["type"], row["source"],
                row["episodes"], row["status"], int(bool(row["airing"])) if pd.notna(row["airing"]) else 0,
                row["score"], row["scored_by"], row["rank"], row["popularity"], row["members"], row["favorites"],
                row["synopsis"], row["rating"], row["year"], row["season"], row["aired_from"], row["aired_to"],
                row["genres"], row["themes"], row["studios"], row["demographics"], row["url"], row["fetched_at"]
            ))
            if cur.rowcount > 0:
                inserted += 1
        except Exception as e:
            print("Raw insert error:", e)

    conn.commit()
    conn.close()
    return inserted


def insert_processed_anime(df_processed, db_name):
    if df_processed.empty:
        return 0

    conn = sqlite3.connect(db_name)
    cur = conn.cursor()
    inserted = 0

    for _, row in df_processed.iterrows():
        try:
            cur.execute("""
                INSERT INTO processed_anime (
                    mal_id, clean_title, clean_title_english, clean_synopsis,
                    title_length, synopsis_length, has_english_title, score_band,
                    popularity_band, episode_group, genre_count, theme_count,
                    studio_count, demographic_count
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row["mal_id"], row["clean_title"], row["clean_title_english"], row["clean_synopsis"],
                row["title_length"], row["synopsis_length"], row["has_english_title"], row["score_band"],
                row["popularity_band"], row["episode_group"], row["genre_count"], row["theme_count"],
                row["studio_count"], row["demographic_count"]
            ))
            inserted += 1
        except Exception as e:
            print("Processed insert error:", e)

    conn.commit()
    conn.close()
    return inserted

